In [3]:
import s3fs
import xarray as xr
import numpy as np
import s3fs

In [4]:
rca_data_bucket = "ooi-data/" # contains data as it comes off the cabled array 
rca_advanced_qaqc_bucket = "rca-advanced-qaqc/" # contains data with additional value-added qaqc 
fs = s3fs.S3FileSystem(anon=True)

In [5]:
def load_data(stream_name, bucket):
    zarr_dir = bucket + stream_name
    zarr_store = fs.get_mapper(zarr_dir)
    ds = xr.open_zarr(zarr_store, consolidated=True)
    return ds

In [6]:
ds = load_data("CE04OSPS-SF01B-4A-NUTNRA102-streamed-nutnr_a_sample/" , rca_data_bucket)

In [7]:
ds

<xarray.Dataset> Size: 975MB
Dimensions:                                     (time: 787210, wavelength: 256)
Coordinates:
  * time                                        (time) datetime64[ns] 6MB 201...
  * wavelength                                  (wavelength) int32 1kB 0 ... 255
Data variables: (12/46)
    aux_fitting_1                               (time) float32 3MB ...
    aux_fitting_2                               (time) float32 3MB ...
    checksum                                    (time) float32 3MB ...
    date_of_sample                              (time) float64 6MB ...
    deployment                                  (time) int32 3MB ...
    driver_timestamp                            (time) datetime64[ns] 6MB ...
    ...                                          ...
    temp_interior                               (time) float32 3MB ...
    temp_lamp                                   (time) float32 3MB ...
    temp_spectrometer                           (time) float32 3MB ...
    time_of_sample                              (time) float64 6MB ...
    voltage_lamp                                (time) float32 3MB ...
    voltage_main                                (time) float32 3MB ...
Attributes: (12/62)
    AssetManagementRecordLastModified:  2026-05-19T15:54:32.531000
    AssetUniqueID:                      ATOSU-68020-00008
    Conventions:                        CF-1.6
    Description:                        Nitrate: NUTNR Series A
    FirmwareVersion:                    Not specified.
    Manufacturer:                       Satlantic
    ...                                 ...
    stream:                             nutnr_a_sample
    subsite:                            CE04OSPS
    summary:                            Dataset Generated by Stream Engine fr...
    time_coverage_end:                  2026-06-01T08:29:41.624592896
    time_coverage_start:                2015-08-03T15:19:44.050183168
    title:                              Data produced by Stream Engine versio...

In [8]:
da = ds.salinity_corrected_nitrate
inf_mask = np.isinf(da).compute()

n_inf = int(inf_mask.sum())
times = da.time.values[inf_mask.values]

print(f"total inf values    : {n_inf:,}")
print(f"time steps affected : {len(times)}")
print()
print("affected timestamps:")
for t in times:
    print(f"  {np.datetime_as_string(t, unit='s')}")

total inf values    : 1
time steps affected : 1

affected timestamps:
  2019-05-13T07:29:07


In [14]:
da.sel(time="2019-05-13T07:29:07").values

array([inf])

In [9]:
qartod = ds.salinity_corrected_nitrate_qartod_results

In [11]:
qartod.sel(time="2019-05-13T07:29:07").values

array([9], dtype=uint8)

In [12]:
ds.salinity_corrected_nitrate_qartod_results

<xarray.DataArray 'salinity_corrected_nitrate_qartod_results' (time: 787210)> Size: 787kB
[787210 values with dtype=uint8]
Coordinates:
  * time     (time) datetime64[ns] 6MB 2015-08-03T15:19:44.050183168 ... 2026...
Attributes:
    comment:        Summary QARTOD test flags. For each datum, the flag is se...
    flag_meanings:  pass not_evaluated suspect_or_of_high_interest fail missi...
    flag_values:    1,2,3,4,9
    long_name:      Nitrate Concentration - Temp and Sal Corrected QARTOD Sum...
    references:     https://ioos.noaa.gov/project/qartod https://github.com/i...
    standard_name:  salinity_corrected_nitrate status_flag